[download this notebook here](https://github.com/HumanCompatibleAI/imitation/blob/master/docs/tutorials/4_train_airl.ipynb)
# Train an Agent using Adversarial Inverse Reinforcement Learning

As usual, we first need an expert. Again, we download one from the HuggingFace model hub for convenience.

Note that we now use a variant of the CartPole environment from the seals package, which has fixed episode durations. Read more about why we do this [here](https://imitation.readthedocs.io/en/latest/getting-started/variable-horizon.html).

In [ ]:
# Imports
import numpy as np
from imitation.policies.serialize import load_policy
from imitation.util.util import make_vec_env
from imitation.data.wrappers import RolloutInfoWrapper
import warnings
warnings.filterwarnings("ignore")

SEED = 42
FAST = False

# Less training steps
if FAST:
    N_RL_TRAIN_STEPS = 100_000
else:
    N_RL_TRAIN_STEPS = 2_000_000

# Load settings like the world, etc.
venv = make_vec_env(
    "seals:seals/CartPole-v0",
    rng=np.random.default_rng(SEED),
    n_envs=8,
    post_wrappers=[
        lambda env, _: RolloutInfoWrapper(env)
    ],  # needed for computing rollouts later
)

# Load agent with baseline policy (rules)
expert = load_policy(
    "ppo-huggingface",
    organization="HumanCompatibleAI",
    env_name="seals/CartPole-v0",
    venv=venv,
)

We generate some expert trajectories, that the discriminator needs to distinguish from the learner's trajectories.

In [26]:
from imitation.data import rollout

# Generate episodes
rollouts = rollout.rollout(
    expert,
    venv,
    rollout.make_sample_until(min_timesteps=None, min_episodes=60),
    rng=np.random.default_rng(SEED),
)

Now we are ready to set up our AIRL trainer.
Note, that the `reward_net` is actually the network of the discriminator.
We evaluate the learner before and after training so we can see if it made any progress.

In [27]:
from imitation.algorithms.adversarial.airl import AIRL
from imitation.rewards.reward_nets import BasicShapedRewardNet
from imitation.util.networks import RunningNorm
from stable_baselines3 import PPO
from stable_baselines3.ppo import MlpPolicy
from stable_baselines3.common.evaluation import evaluate_policy

# PPO: Rule updater (updates policies)
learner = PPO(
    env=venv,
    policy=MlpPolicy,
    batch_size=1024,
    ent_coef=0.0,
    learning_rate=0.0005,
    gamma=0.95,
    clip_range=0.1,
    vf_coef=0.1,
    n_epochs=5,
    seed=SEED,
)

# Empty reward function
reward_net = BasicShapedRewardNet(
    observation_space=venv.observation_space,
    action_space=venv.action_space,
    normalize_input_layer=RunningNorm,
)

# AIRL (Gen + Disc IRL)
airl_trainer = AIRL(
    demonstrations=rollouts,
    demo_batch_size=4096,
    gen_replay_buffer_capacity=512,
    n_disc_updates_per_round=16,
    venv=venv,
    gen_algo=learner,
    reward_net=reward_net,
)

# Extract policy beforehand
venv.seed(SEED)
rewards_before, steps_before = evaluate_policy(
    learner, venv, 100, return_episode_rewards=True
)

# Gen and Disc fight it out
airl_trainer.train(N_RL_TRAIN_STEPS)

# 
venv.seed(SEED)
rewards_after, steps_after = evaluate_policy(
    learner, venv, 100, return_episode_rewards=True
)

round:   0%|          | 0/122 [00:00<?, ?it/s]

------------------------------------------
| raw/                        |          |
|    gen/rollout/ep_len_mean  | 500      |
|    gen/rollout/ep_rew_mean  | 34.4     |
|    gen/time/fps             | 2742     |
|    gen/time/iterations      | 1        |
|    gen/time/time_elapsed    | 5        |
|    gen/time/total_timesteps | 16384    |
------------------------------------------
--------------------------------------------------
| raw/                                |          |
|    disc/disc_acc                    | 0.505    |
|    disc/disc_acc_expert             | 1        |
|    disc/disc_acc_gen                | 0.00952  |
|    disc/disc_entropy                | 0.663    |
|    disc/disc_loss                   | 0.738    |
|    disc/disc_proportion_expert_pred | 0.995    |
|    disc/disc_proportion_expert_true | 0.5      |
|    disc/global_step                 | 1        |
|    disc/n_expert                    | 4.1e+03  |
|    disc/n_generated                 | 4.1e+03  |
-

round:   1%|          | 1/122 [00:07<14:07,  7.00s/it]

-----------------------------------------------------
| raw/                               |              |
|    gen/rollout/ep_len_mean         | 500          |
|    gen/rollout/ep_rew_mean         | 36.3         |
|    gen/rollout/ep_rew_wrapped_mean | -572         |
|    gen/time/fps                    | 3190         |
|    gen/time/iterations             | 1            |
|    gen/time/time_elapsed           | 5            |
|    gen/time/total_timesteps        | 32768        |
|    gen/train/approx_kl             | 0.0026358052 |
|    gen/train/clip_fraction         | 0.0219       |
|    gen/train/clip_range            | 0.1          |
|    gen/train/entropy_loss          | -0.692       |
|    gen/train/explained_variance    | -0.014631152 |
|    gen/train/learning_rate         | 0.0005       |
|    gen/train/loss                  | 31.8         |
|    gen/train/n_updates             | 5            |
|    gen/train/policy_gradient_loss  | -0.000624    |
|    gen/train/value_loss   

round:   2%|▏         | 2/122 [00:12<12:48,  6.41s/it]

...

round:  98%|█████████▊| 120/122 [12:01<00:12,  6.32s/it]

------------------------------------------------------
| raw/                               |               |
|    gen/rollout/ep_len_mean         | 500           |
|    gen/rollout/ep_rew_mean         | 500           |
|    gen/rollout/ep_rew_wrapped_mean | -53.3         |
|    gen/time/fps                    | 3610          |
|    gen/time/iterations             | 1             |
|    gen/time/time_elapsed           | 4             |
|    gen/time/total_timesteps        | 1982464       |
|    gen/train/approx_kl             | 0.00072186725 |
|    gen/train/clip_fraction         | 0.0209        |
|    gen/train/clip_range            | 0.1           |
|    gen/train/entropy_loss          | -0.283        |
|    gen/train/explained_variance    | 0.9643284     |
|    gen/train/learning_rate         | 0.0005        |
|    gen/train/loss                  | 0.312         |
|    gen/train/n_updates             | 600           |
|    gen/train/policy_gradient_loss  | -0.00143      |
|    gen/t

round:  99%|█████████▉| 121/122 [12:07<00:06,  6.09s/it]

-----------------------------------------------------
| raw/                               |              |
|    gen/rollout/ep_len_mean         | 500          |
|    gen/rollout/ep_rew_mean         | 500          |
|    gen/rollout/ep_rew_wrapped_mean | 12.2         |
|    gen/time/fps                    | 3151         |
|    gen/time/iterations             | 1            |
|    gen/time/time_elapsed           | 5            |
|    gen/time/total_timesteps        | 1998848      |
|    gen/train/approx_kl             | 0.0004625126 |
|    gen/train/clip_fraction         | 0.0106       |
|    gen/train/clip_range            | 0.1          |
|    gen/train/entropy_loss          | -0.273       |
|    gen/train/explained_variance    | 0.95610845   |
|    gen/train/learning_rate         | 0.0005       |
|    gen/train/loss                  | 0.324        |
|    gen/train/n_updates             | 605          |
|    gen/train/policy_gradient_loss  | -0.0012      |
|    gen/train/value_loss   

round: 100%|██████████| 122/122 [12:13<00:00,  6.01s/it]


We can see that an untrained policy performs poorly, while AIRL brings an improvement. To make it match the expert performance (500), set the flag `FAST` to `False` in the first cell.

In [28]:
print(
    "Rewards before training:",
    np.mean(rewards_before),
    "+/-",
    np.std(rewards_before),
)
print(
    "Rewards after training:",
    np.mean(rewards_after),
    "+/-",
    np.std(rewards_after),
)

Rewards before training: 102.6 +/- 24.11514047232568
Rewards after training: 500.0 +/- 0.0


In [29]:
import pandas as pd

num_episodes = len(rewards_after)
# print(num_episodes)

# 2. Structure the data into a dictionary
metrics_data = {
    "Episode": np.arange(1, num_episodes + 1),
    "Total Reward": rewards_after,
    "Episode Length (Steps)": steps_after,
}

# 3. Create your table using Pandas
df_metrics = pd.DataFrame(metrics_data)

# Add a summary row at the bottom for your "bottom line" stats
summary_row = pd.DataFrame({
    "Episode": ["AVERAGE"],
    "Total Reward": [np.mean(rewards_after)],
    "Episode Length (Steps)": [np.mean(steps_after)]
})

df_metrics = pd.concat([df_metrics, summary_row], ignore_index=True)

# Print your beautiful table
df_metrics

,Episode,Total Reward,Episode Length (Steps)
0,1,500.0,500.0
1,2,500.0,500.0
2,3,500.0,500.0
3,4,500.0,500.0
4,5,500.0,500.0
...,...,...,...
96,97,500.0,500.0
97,98,500.0,500.0
98,99,500.0,500.0
99,100,500.0,500.0
